<h1> GAIA TELESCOPE — Intelligent Query Pipeline</h1>
Guillem Masdemont, Plabon Shaha, Pietro Sestito 

## 1 · Environment setup & imports

In [1]:
# ── SSL fix for ARNES cluster ─────────────────────────────────────────────────
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import os
os.environ.setdefault('CURL_CA_BUNDLE', '')
os.environ.setdefault('REQUESTS_CA_BUNDLE', '')

# ── Standard imports ──────────────────────────────────────────────────────────
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import plotly.graph_objects as go
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='darkgrid')

print('Imports OK')

Imports OK


## 2 · HuggingFace cache & SSL

In [2]:
HF_CACHE = os.path.abspath('../hf_cache')
os.environ['HF_HOME']            = HF_CACHE
os.environ['HF_HUB_CACHE']       = os.path.join(HF_CACHE, 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'

# SSL fixes (repeat after imports to be safe)
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()
os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

print(f'HF cache: {HF_CACHE}')

HF cache: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache


## 3 · Load vLLM model

In [3]:
from vllm import LLM, SamplingParams

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
TENSOR_PARALLEL_SIZE = int(os.environ.get('TENSOR_PARALLEL_SIZE', '1'))
DTYPE = 'float16'

if 'llm' not in globals():
    print(f'Loading model from {HF_CACHE} …')
    llm = LLM(
        model=MODEL_ID,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        dtype=DTYPE,
        gpu_memory_utilization=0.90,
        trust_remote_code=True,
    )
    print('✅  Model loaded.')
else:
    print('Model already loaded — reusing.')

# Sampling params for the query parser (deterministic)
SAMPLING_PARAMS       = SamplingParams(temperature=0.0, max_tokens=256)
# Sampling params for the cost judge (needs more tokens to reason)
JUDGE_SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=512)

/usr/local/lib/python3.11/dist-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading model from /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache …


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

WARNING 05-01 11:58:01 config.py:2276] Casting torch.bfloat16 to torch.float16.
INFO 05-01 11:58:15 config.py:510] This model supports multiple tasks: {'reward', 'score', 'embed', 'classify', 'generate'}. Defaulting to 'generate'.
INFO 05-01 11:58:15 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forwa

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

INFO 05-01 11:58:19 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 05-01 11:58:19 selector.py:129] Using XFormers backend.
INFO 05-01 11:58:21 model_runner.py:1094] Starting to load model Qwen/Qwen2.5-7B-Instruct...
INFO 05-01 11:58:27 weight_utils.py:251] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-01 12:01:53 model_runner.py:1099] Loading model weights took 14.2487 GB
INFO 05-01 12:02:02 worker.py:241] Memory profiling takes 8.74 seconds
INFO 05-01 12:02:02 worker.py:241] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.90) = 28.56GiB
INFO 05-01 12:02:02 worker.py:241] model weights take 14.25GiB; non_torch_memory takes 0.13GiB; PyTorch activation peak memory takes 4.35GiB; the rest of the memory reserved for KV Cache is 9.82GiB.
INFO 05-01 12:02:02 gpu_executor.py:76] # GPU blocks: 11497, # CPU blocks: 4681
INFO 05-01 12:02:02 gpu_executor.py:80] Maximum concurrency for 32768 tokens per request: 5.61x
INFO 05-01 12:02:08 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utiliz

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:37<00:00,  1.08s/it]

INFO 05-01 12:02:46 model_runner.py:1535] Graph capturing finished in 38 secs, took 0.20 GiB
INFO 05-01 12:02:46 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 52.85 seconds
✅  Model loaded.


## 4 · LLM query parser & ADQL builder

In [ ]:
Position (ra, dec): The precise sky coordinates.
Parallax (parallax): Used to calculate the distance to the star.
Proper Motion (pmra, pmdec): The 2D velocity of the star across the sky.
Radial Velocity (radial_velocity): The speed toward or away from the observer.
G-mag (phot_g_mean_mag): Total brightness in the broad Gaia filter.
BP and RP (phot_bp_mean_mag, phot_rp_mean_mag): Blue and Red photometer magnitudes.
Effective Temperature (teff_gspphot): Surface temperature in Kelvin.
Surface Gravity (logg_gspphot): Indicates if the star is a giant or a dwarf.
Color Index (bp_rp): The difference between BP and RP magnitudes.
Photometric Variability: Flags or data indicating if the star's brightness changes over time.

In [23]:
# ── System prompt for the query parser LLM ───────────────────────────────────
SYSTEM_PROMPT = """You are an astronomy query parser.

Convert user queries into structured JSON. 

Supported intents:
- cone_search
- cone_search_with_join
- color_histogram
- velocity_computation

Return ONLY valid JSON — no markdown fences, no extra text.

Schema:
{
  "intent": "...",
  "ra": float or null,
  "dec": float or null,
  "radius": float,
  "columns": [list],
  "limit": int
}

Valid Gaia DR3 column names (use ONLY these):
  ra, dec, source_id, parallax, parallax_error,
  pmra, pmdec, radial_velocity,
  phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp,
  teff_gspphot, logg_gspphot, mh_gspphot

Rules:
- If coordinates are unknown use: ra=266.4, dec=-29.0 (galactic centre)
- For velocity_computation without a specific sky region, ra and dec MAY be null
- Default radius: 1.0 degree
- Default limit: 1000"""


def _build_prompt(user_query: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{user_query}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def parse_query(user_query: str) -> dict:
    """Parse a natural-language astronomy query into structured JSON using the LLM."""
    prompt  = _build_prompt(user_query)
    outputs = llm.generate([prompt], SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    # Strip markdown fences if model disobeys
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f'Model returned non-JSON output:\n{raw}') from exc


# ── Column alias map ──────────────────────────────────────────────────────────
_COLUMN_ALIASES = {
    'magnitude':         'phot_g_mean_mag',
    'g_magnitude':       'phot_g_mean_mag',
    'g_mag':             'phot_g_mean_mag',
    'bp_magnitude':      'phot_bp_mean_mag',
    'rp_magnitude':      'phot_rp_mean_mag',
    'color':             'bp_rp',
    'colour':            'bp_rp',
    'temperature':       'teff_gspphot',
    'teff':              'teff_gspphot',
    'proper_motion_ra':  'pmra',
    'proper_motion_dec': 'pmdec',
}


def build_adql(q: dict) -> str:
    """Convert a parsed JSON dict into an ADQL query string."""
    intent  = q['intent']
    ra      = q.get('ra')
    dec     = q.get('dec')
    radius  = q.get('radius', 1.0)
    limit   = q.get('limit', 1000)
    columns = [_COLUMN_ALIASES.get(c, c) for c in (q.get('columns') or ['ra', 'dec', 'parallax'])]

    # Spatial cone — only built when we have real coordinates
    def cone(ra_col='ra', dec_col='dec'):
        return f"1=CONTAINS(POINT('ICRS', {ra_col}, {dec_col}), CIRCLE('ICRS', {ra}, {dec}, {radius}))"

    if intent == 'cone_search':
        cols = ', '.join(columns)
        return f"SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {cone()}"

    elif intent == 'cone_search_with_join':
        return (
            f"SELECT TOP {limit} source_id, g.source_id, g.ra, g.dec, d.r_med_geo "
            f"FROM gaiadr3.gaia_source AS g "
            f"JOIN gaiadr3.geometric_distance_prior AS d ON g.source_id = d.source_id "
            f"WHERE {cone('g.ra', 'g.dec')}"
        )

    elif intent == 'color_histogram':
        return (
            f"SELECT TOP {limit} source_id, ra, dec, bp_rp, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag "
            f"FROM gaiadr3.gaia_source "
            f"WHERE bp_rp IS NOT NULL AND {cone()}"
        )

    elif intent == 'velocity_computation':
        # Full-sky closest-stars query when no coordinates given
        if ra is None or dec is None:
            return (
                f"SELECT TOP {limit} source_id, ra, dec, parallax, pmra, pmdec, radial_velocity "
                f"FROM gaiadr3.gaia_source "
                f"WHERE parallax > 100 "
                f"AND radial_velocity IS NOT NULL "
                f"ORDER BY parallax DESC"
            )
        else:
            return (
                f"SELECT TOP {limit} source_id, ra, dec, parallax, pmra, pmdec, radial_velocity "
                f"FROM gaiadr3.gaia_source "
                f"WHERE parallax > 10 "
                f"AND radial_velocity IS NOT NULL "
                f"AND {cone()} "
                f"ORDER BY parallax DESC"
            )

    else:
        raise ValueError(f"Unknown intent: '{intent}'")

## 5 · Gaia TAP query runner

In [5]:
def run_query(adql: str, retries: int = 3, backoff: float = 15.0, row_limit: int = 50_000) -> pd.DataFrame:
    """
    Submit an ADQL query to the Gaia TAP service and return a pandas DataFrame.
    Automatically injects TOP N if the query has no row cap.
    Fails fast on HTTP 500 (bad query) instead of burning retries.
    """
    from astroquery.gaia import Gaia

    Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'
    Gaia.ROW_LIMIT       = row_limit

    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        adql = re.sub(r'\bSELECT\b', f'SELECT TOP {row_limit}',
                      adql, count=1, flags=re.IGNORECASE)
        print(f'Injected TOP {row_limit}')

    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        print('No WHERE clause — query may time out.')

    print(f'\nQuery sent:\n{adql}\n')

    last_exc = None
    for attempt in range(retries):
        try:
            job    = Gaia.launch_job_async(adql, verbose=False)
            result = job.get_results()
            print(f'Got {len(result):,} rows')
            return result.to_pandas()
        except Exception as exc:
            last_exc = exc
            print(f'[attempt {attempt+1}/{retries}] {exc}')
            if '500' in str(exc):
                print('HTTP 500 — bad query syntax, not a transient error.')
                raise exc
            if attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f'    Waiting {wait:.0f}s …')
                time.sleep(wait)
    raise last_exc

## 6 · Teacher — JSON & ADQL validator

In [6]:
MAX_RETRIES = 3

VALID_COLUMNS = {
    'ra', 'dec', 'source_id', 'parallax', 'parallax_error',
    'pmra', 'pmdec', 'radial_velocity',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp',
    'teff_gspphot', 'logg_gspphot', 'mh_gspphot',
}

VALID_INTENTS = {
    'cone_search', 'cone_search_with_join',
    'color_histogram', 'velocity_computation',
}


def validate_parsed_json(q: dict) -> list:
    """Return list of error strings. Empty = valid."""
    errors  = []
    intent  = q.get('intent')

    if not intent or intent not in VALID_INTENTS:
        errors.append(
            f"'intent' is '{intent}', must be one of: {sorted(VALID_INTENTS)}."
        )

    for field in ('ra', 'dec'):
        val = q.get(field)
        # velocity_computation is allowed to have null coords (full-sky)
        if val is None and intent == 'velocity_computation':
            continue
        if val is None:
            errors.append(
                f"'{field}' is null. Provide numeric degrees or use galactic centre "
                f"(ra=266.4, dec=-29.0) as default."
            )
        elif not isinstance(val, (int, float)):
            errors.append(f"'{field}' must be a number, got: {repr(val)}.")
        elif field == 'ra'  and not (0 <= float(val) <= 360):
            errors.append(f"'ra' = {val} is outside [0, 360].")
        elif field == 'dec' and not (-90 <= float(val) <= 90):
            errors.append(f"'dec' = {val} is outside [-90, 90].")

    radius = q.get('radius')
    if radius is None:
        errors.append("'radius' is null. Use default 1.0 degree.")
    elif not isinstance(radius, (int, float)) or float(radius) <= 0:
        errors.append(f"'radius' must be positive, got: {repr(radius)}.")
    elif float(radius) > 10:
        errors.append(f"'radius' = {radius}° is very large (> 10°) and will likely time out.")

    limit = q.get('limit', 1000)
    if not isinstance(limit, int) or limit <= 0:
        errors.append(f"'limit' must be a positive integer, got: {repr(limit)}.")
    elif limit > 100_000:
        errors.append(f"'limit' = {limit:,} is too large. Use < 100,000.")

    cols    = q.get('columns') or []
    unknown = [c for c in cols if c not in VALID_COLUMNS and c not in _COLUMN_ALIASES]
    if unknown:
        errors.append(
            f"Unknown column(s): {unknown}. "
            f"Allowed: {sorted(VALID_COLUMNS)}."
        )

    return errors


def validate_adql(adql: str) -> list:
    """Catch problems in the final ADQL string."""
    errors = []
    if re.search(r'\bNone\b', adql):
        errors.append(
            "ADQL contains literal 'None' — a coordinate was never resolved. "
            "Replace with actual degree values."
        )
    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        errors.append('No TOP clause — add TOP N to cap result size.')
    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        errors.append('No WHERE clause — add a spatial or quality filter.')
    return errors


def build_correction_prompt(original_query: str, bad_json: dict,
                             errors: list, attempt: int) -> str:
    error_block = '\n'.join(f'  - {e}' for e in errors)
    return (
        f'Your previous answer for the query:\n'
        f'  "{original_query}"\n\n'
        f'produced this JSON:\n'
        f'{json.dumps(bad_json, indent=2)}\n\n'
        f'The teacher found {len(errors)} error(s) (attempt {attempt}/{MAX_RETRIES}):\n'
        f'{error_block}\n\n'
        f'Fix ALL errors above and return ONLY corrected JSON. '
        f'No markdown, no explanation.'
    )


## 7 · Cost judge — LLM-based query expense evaluator

In [12]:
COST_JUDGE_SYSTEM_PROMPT = """You are an expert in ADQL queries on the Gaia DR3 database (1.8 billion rows).
Evaluate how expensive a query will be on the Gaia TAP server.

Return ONLY valid JSON, no markdown:
{
  "verdict":       "cheap" | "moderate" | "expensive" | "dangerous",
  "score":         0-100,
  "reasons":       [list of strings],
  "optimisations": [list of concrete ADQL rewrites],
  "safe_to_run":   true | false
}

Scoring guide:
  0-25  cheap     : TOP ≤ 1000, tight cone ≤ 0.5°, indexed filter, no JOIN
  26-50 moderate  : TOP ≤ 10000, cone ≤ 2°, simple filters
  51-75 expensive : TOP ≤ 50000, cone > 2°, ORDER BY non-indexed col, weak parallax
  76-100 dangerous: no TOP or TOP > 100000, no spatial filter, full-table scan, multi-JOIN

Key cost drivers:
  - Missing/large TOP (biggest risk)
  - Cone radius > 2 degrees
  - ORDER BY on parallax or radial_velocity (not indexed)
  - radial_velocity IS NOT NULL  (only ~1% of stars, forces full scan)
  - parallax > 0  (matches ~90% of rows — very weak)
  - Multiple JOINs
  - No WHERE clause

Optimisation strategies:
  - Reduce TOP N
  - Shrink cone radius
  - Replace parallax > 0 with parallax > 10 (nearby stars)
  - Replace parallax > 0 with parallax > 100 (closest stars, < 10 pc)
  - Add phot_g_mean_mag < 15 to cut faint stars early
  - Remove ORDER BY or replace with a parallax threshold filter"""


def _build_cost_prompt(adql: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{COST_JUDGE_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'Evaluate this ADQL query:\n\n{adql}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def fast_cost_precheck(adql: str):
    """Rule-based fast check — no LLM call. Returns dict if dangerous, else None."""
    adql_upper = adql.upper()
    if 'CONTAINS' not in adql_upper and 'WHERE' not in adql_upper:
        return {
            'verdict': 'dangerous', 'score': 100, 'safe_to_run': False,
            'reasons': ['No WHERE clause — full scan of 1.8B rows.'],
            'optimisations': ['Add a CIRCLE spatial filter and TOP clause.'],
        }
    top_match = re.search(r'\bTOP\s+(\d+)\b', adql, re.IGNORECASE)
    if not top_match:
        return {
            'verdict': 'dangerous', 'score': 95, 'safe_to_run': False,
            'reasons': ['No TOP clause — unbounded result set.'],
            'optimisations': ['Add TOP 1000 after SELECT.'],
        }
    if int(top_match.group(1)) > 100_000:
        return {
            'verdict': 'dangerous', 'score': 90, 'safe_to_run': False,
            'reasons': [f'TOP {int(top_match.group(1)):,} exceeds 100,000 row limit.'],
            'optimisations': ['Reduce TOP to ≤ 10,000.'],
        }
    return None


def evaluate_query_cost(adql: str) -> dict:
    """LLM judge: score and explain query cost."""
    prompt  = _build_cost_prompt(adql)
    outputs = llm.generate([prompt], JUDGE_SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'Cost judge returned non-JSON:\n{raw}')
        return {
            'verdict': 'expensive', 'score': 70, 'safe_to_run': False,
            'reasons': ['Cost judge failed to parse — defaulting conservative.'],
            'optimisations': [],
        }


def auto_optimise_adql(adql: str, optimisations: list, top_cap: int = 5_000) -> str:
    """Apply deterministic optimisations to the ADQL string."""
    # Cap TOP N
    def cap_top(m):
        n = int(m.group(1))
        if n > top_cap:
            print(f'TOP {n:,} → {top_cap:,}')
            return f'TOP {top_cap}'
        return m.group(0)
    adql = re.sub(r'\bTOP\s+(\d+)\b', cap_top, adql, flags=re.IGNORECASE)

    # Shrink cones > 3° to 2°
    def shrink_cone(m):
        if float(m.group(3)) > 3.0:
            print(f'Cone {m.group(3)}° → 2.0°')
            return f"CIRCLE('ICRS', {m.group(1)}, {m.group(2)}, 2.0)"
        return m.group(0)
    adql = re.sub(
        r"CIRCLE\s*\(\s*'ICRS'\s*,\s*([0-9.\-]+)\s*,\s*([0-9.\-]+)\s*,\s*([0-9.]+)\s*\)",
        shrink_cone, adql, flags=re.IGNORECASE
    )

    # Strengthen weak parallax filter
    before = adql
    adql = re.sub(r'\bparallax\s*>\s*0\b', 'parallax > 1', adql, flags=re.IGNORECASE)
    if adql != before:
        print('parallax > 0  →  parallax > 1')

    if optimisations:
        print('LLM judge also suggested:')
        for opt in optimisations:
            print(f'       • {opt}')
    return adql


_VERDICT_EMOJI = {'cheap': '🟢', 'moderate': '🟡', 'expensive': '🟠', 'dangerous': '🔴'}


def print_cost_report(cost: dict):
    verdict = cost.get('verdict', '?')
    score   = cost.get('score', '?')
    emoji   = _VERDICT_EMOJI.get(verdict, '⚪')
    print(f"\n  {'─'*52}")
    print(f"  {emoji}  COST: {verdict.upper()}  (score {score}/100)")
    print(f"  {'─'*52}")
    for r in cost.get('reasons', []):
        print(f'    ⚠️  {r}')
    for o in cost.get('optimisations', []):
        print(f'    💡 {o}')
    print(f"  {'─'*52}\n")

## 8 · Supervised pipeline — everything wired together

In [13]:
def supervised_pipeline(user_query: str) -> pd.DataFrame:
    """
    Full pipeline:
      1. Optional geographic direction resolution
      2. LLM parses query → JSON
      3. Teacher validates JSON  (correction loop)
      4. build_adql()
      5. Teacher validates ADQL  (correction loop)
      6. Cost judge              (block / optimise / approve)
      7. run_query()             (correction loop on 500)

    Raises RuntimeError("This query is too difficult for me") after MAX_RETRIES.
    """
    print(f"\n{'='*60}")
    print(f'  USER QUERY: {user_query}')
    print(f"{'='*60}\n")

    current_prompt = user_query

    parsed = None
    adql   = None

    for attempt in range(1, MAX_RETRIES + 1):

        # ── Stage 1: LLM → JSON ──────────────────────────────────────────────
        print(f'[Attempt {attempt}/{MAX_RETRIES}] Parsing …')
        try:
            parsed = parse_query(current_prompt)
        except ValueError as exc:
            print(f'Non-JSON from LLM: {exc}')
            current_prompt = (
                f'Your answer was not valid JSON.\n'
                f'Original query: "{user_query}"\n'
                f'Error: {exc}\n'
                f'Return ONLY a valid JSON object.'
            )
            continue

        print(f'Parsed JSON:\n{json.dumps(parsed, indent=2)}')

        # ── Stage 2: Teacher validates JSON ──────────────────────────────────
        json_errors = validate_parsed_json(parsed)
        if json_errors:
            print(f'JSON errors ({len(json_errors)}):')
            for e in json_errors:
                print(f'     • {e}')
            current_prompt = build_correction_prompt(user_query, parsed, json_errors, attempt)
            continue

        print('JSON valid.')

        # ── Stage 3: Build ADQL ───────────────────────────────────────────────
        try:
            adql = build_adql(parsed)
        except (ValueError, KeyError) as exc:
            print(f'build_adql() failed: {exc}')
            current_prompt = build_correction_prompt(
                user_query, parsed, [f'build_adql() error: {exc}'], attempt
            )
            continue

        print(f'  ADQL:\n  {adql}')

        # ── Stage 4: Teacher validates ADQL ──────────────────────────────────
        adql_errors = validate_adql(adql)
        if adql_errors:
            print(f'ADQL errors ({len(adql_errors)}):')
            for e in adql_errors:
                print(f'     • {e}')
            current_prompt = build_correction_prompt(user_query, parsed, adql_errors, attempt)
            continue

        print('ADQL syntax valid.')

        # ── Stage 5: Cost gate ────────────────────────────────────────────────
        print('Evaluating cost …')
        cost = fast_cost_precheck(adql)
        if cost is None:
            cost = evaluate_query_cost(adql)

        print_cost_report(cost)

        if cost['verdict'] == 'dangerous':
            print('  🔴 BLOCKED — query too dangerous.')
            current_prompt = build_correction_prompt(
                user_query, parsed,
                [f"Query rated DANGEROUS (score {cost['score']}/100). "
                 f"Reasons: {'; '.join(cost['reasons'])}. "
                 f"Fix: {'; '.join(cost['optimisations'])}"],
                attempt,
            )
            continue

        if cost['verdict'] == 'expensive':
            print('  🟠 Expensive — auto-optimising …')
            adql  = auto_optimise_adql(adql, cost.get('optimisations', []))
            print(f'  Optimised ADQL:\n  {adql}')
            cost2 = fast_cost_precheck(adql) or evaluate_query_cost(adql)
            print_cost_report(cost2)
            if cost2['verdict'] == 'dangerous':
                print('  🔴 Still dangerous after optimisation — blocking.')
                current_prompt = build_correction_prompt(
                    user_query, parsed,
                    [f"Still dangerous after auto-optimisation. Reasons: {'; '.join(cost2['reasons'])}"],
                    attempt,
                )
                continue

        # ── Stage 6: Execute ──────────────────────────────────────────────────
        try:
            df = run_query(adql)
            print(f'\n✅  Pipeline done on attempt {attempt}. {len(df):,} rows.\n')
            return df
        except Exception as exc:
            print(f'  ❌ run_query() failed: {exc}')
            current_prompt = build_correction_prompt(
                user_query, parsed,
                [f'Gaia TAP error: {exc}. Check column names and ADQL syntax.'],
                attempt,
            )
            continue

    raise RuntimeError(
        f'This query is too difficult for me.\n'
        f'Last JSON: {json.dumps(parsed, indent=2)}\n'
        f'Last ADQL: {adql}'
    )


print('✅  supervised_pipeline defined')

✅  supervised_pipeline defined


## 9 · 3-D Celestial Sphere visualisation

In [14]:
def radec_to_xyz(ra_deg: np.ndarray, dec_deg: np.ndarray):
    """RA/Dec (degrees) → Cartesian (x, y, z) on the unit sphere."""
    ra  = np.radians(ra_deg)
    dec = np.radians(dec_deg)
    return np.cos(dec)*np.cos(ra), np.cos(dec)*np.sin(ra), np.sin(dec)


def _sphere_wireframe(n: int = 36):
    """RA/Dec grid lines for the reference sphere."""
    u = np.linspace(0, 2*np.pi, n)
    v = np.linspace(-np.pi/2, np.pi/2, n//2)
    lx, ly, lz = [], [], []
    for vi in v[::3]:                                  # parallels
        lx += [*np.cos(vi)*np.cos(u), None]
        ly += [*np.cos(vi)*np.sin(u), None]
        lz += [*np.full_like(u, np.sin(vi)), None]
    for ui in u[::3]:                                  # meridians
        lx += [*np.cos(v)*np.cos(ui), None]
        ly += [*np.cos(v)*np.sin(ui), None]
        lz += [*np.sin(v), None]
    return lx, ly, lz


def plot_celestial_sphere(df: pd.DataFrame, save_html: str = None):
    """
    Interactive 3-D celestial sphere.  Drag to rotate, scroll to zoom.
    Colour-codes by G magnitude if available.
    """
    if 'ra' not in df.columns or 'dec' not in df.columns:
        print('ℹ️  No ra/dec columns — nothing to plot.')
        return

    ra  = df['ra'].to_numpy(float)
    dec = df['dec'].to_numpy(float)
    x, y, z = radec_to_xyz(ra, dec)

    colour_col = 'phot_g_mean_mag' if 'phot_g_mean_mag' in df.columns else None
    if colour_col:
        cv = df[colour_col].to_numpy(float)
        mk = dict(
            color=cv, colorscale='Viridis', reversescale=True,
            colorbar=dict(
            title=dict(text='G mag', font=dict(color='#333333', size=10)),
            thickness=10,
            len=0.45,
            x=1.01,
            tickfont=dict(color='#333333', size=8),
            bgcolor='rgba(0,0,0,0)',
            bordercolor='rgba(0,0,0,0)',),
            showscale=True,
        )
        hover  = 'RA: %{customdata[0]:.4f}°<br>Dec: %{customdata[1]:.4f}°<br>G mag: %{customdata[2]:.2f}<extra></extra>'
        custom = np.stack([ra, dec, cv], axis=1)
    else:
        mk     = dict(color='#7ec8e3', showscale=False)
        hover  = 'RA: %{customdata[0]:.4f}°<br>Dec: %{customdata[1]:.4f}°<extra></extra>'
        custom = np.stack([ra, dec], axis=1)

    stars = go.Scatter3d(
        x=x, y=y, z=z, mode='markers',
        marker=dict(size=1.8, opacity=0.85, **mk),
        hovertemplate=hover, customdata=custom, name='Stars',
    )

    wx, wy, wz = _sphere_wireframe()
    grid = go.Scatter3d(
        x=wx, y=wy, z=wz, mode='lines',
        line=dict(color='rgba(100,100,160,0.18)', width=1),
        hoverinfo='skip', name='Grid', showlegend=False,
    )

    eq_u    = np.linspace(0, 2*np.pi, 200)
    equator = go.Scatter3d(
        x=np.cos(eq_u), y=np.sin(eq_u), z=np.zeros(200), mode='lines',
        line=dict(color='rgba(180,140,80,0.45)', width=2),
        hoverinfo='skip', name='Equator', showlegend=False,
    )

    ax = dict(showbackground=False, showgrid=False, zeroline=False,
              showticklabels=False, title='')
    layout = go.Layout(
        title=dict(text=f'Celestial Sphere — {len(df):,} sources',
                   font=dict(color='#dde0ff', size=18, family='Georgia, serif'), x=0.5),
        paper_bgcolor='#05050f',
        scene=dict(xaxis=ax, yaxis=ax, zaxis=ax, bgcolor='#05050f',
                   camera=dict(eye=dict(x=1.4, y=1.4, z=0.6)), aspectmode='cube'),
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(font=dict(color='#aaaacc'), bgcolor='rgba(0,0,0,0)'),
    )

    fig = go.Figure(data=[grid, equator, stars], layout=layout)
    if save_html:
        fig.write_html(save_html, include_plotlyjs='cdn')
        print(f'✅  Saved to {save_html}')
    fig.show()

## 10 · Run a query

Change `USER_QUERY` to any natural language astronomy question.

In [24]:
# ── Pick a query ──────────────────────────────────────────────────────────────
#USER_QUERY = 'How fast do the closest stars move?'

# More examples:
#USER_QUERY = 'Do a cone search around Andromeda with a radius of 0.05 degrees'
#USER_QUERY = 'What color are the stars near the galactic center?'
USER_QUERY = 'Show me the red dwarfs, use a color histogram'
# USER_QUERY = 'Stars near Orion within 0.5 degrees'
# USER_QUERY = 'Stars I can see pointing towards Morocco from Barcelona'   # geographic!
# USER_QUERY = 'Show me stars in the direction of Tokyo from Barcelona'

try:
    df = supervised_pipeline(USER_QUERY)
    display(df.head(10))
except RuntimeError as e:
    print(f'\n🔴  {e}')


  USER QUERY: Show me the red dwarfs, use a color histogram

[Attempt 1/3] Parsing …


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it, est. speed input: 187.75 toks/s, output: 42.93 toks/s]


Parsed JSON:
{
  "intent": "color_histogram",
  "ra": 266.4,
  "dec": -29.0,
  "radius": 1.0,
  "columns": [
    "bp_rp"
  ],
  "limit": 1000
}
JSON valid.
  ADQL:
  SELECT TOP 1000 ra, dec, bp_rp, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag FROM gaiadr3.gaia_source WHERE bp_rp IS NOT NULL AND 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.4, -29.0, 1.0))
ADQL syntax valid.
Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it, est. speed input: 258.35 toks/s, output: 43.54 toks/s]



  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  Tight cone ≤ 1.0°
    ⚠️  Indexed filter (ra, dec)
    ⚠️  No JOIN
    💡 No further optimisations needed
  ────────────────────────────────────────────────────


Query sent:
SELECT TOP 1000 ra, dec, bp_rp, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag FROM gaiadr3.gaia_source WHERE bp_rp IS NOT NULL AND 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.4, -29.0, 1.0))

500 Error 500:
null
[attempt 1/3] Error 500:
null
HTTP 500 — bad query syntax, not a transient error.
  ❌ run_query() failed: Error 500:
null
[Attempt 2/3] Parsing …


...

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.45s/it, est. speed input: 259.36 toks/s, output: 43.34 toks/s]


Parsed JSON:
{
  "intent": "color_histogram",
  "ra": 266.4,
  "dec": -29.0,
  "radius": 1.0,
  "columns": [
    "bp_rp"
  ],
  "limit": 1000
}
JSON valid.
  ADQL:
  SELECT TOP 1000 ra, dec, bp_rp, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag FROM gaiadr3.gaia_source WHERE bp_rp IS NOT NULL AND 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.4, -29.0, 1.0))
ADQL syntax valid.
Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.08s/it, est. speed input: 259.32 toks/s, output: 43.70 toks/s]



  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  Tight cone ≤ 1.0°
    ⚠️  Indexed filter (ra, dec)
    ⚠️  No JOIN
    💡 No further optimisations needed
  ────────────────────────────────────────────────────


Query sent:
SELECT TOP 1000 ra, dec, bp_rp, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag FROM gaiadr3.gaia_source WHERE bp_rp IS NOT NULL AND 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.4, -29.0, 1.0))

INFO: Query finished. [astroquery.utils.tap.core]
Got 1,000 rows

✅  Pipeline done on attempt 3. 1,000 rows.



,ra,dec,bp_rp,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag
0,266.995538,-29.854833,2.330221,17.168329,18.348711,16.018490
1,267.008073,-29.845645,4.643501,18.989344,21.939722,17.296221
2,267.015515,-29.841614,2.196457,18.222445,19.361549,17.165092
3,266.707131,-29.963471,2.222948,18.946039,20.123316,17.900368
4,266.707648,-29.959511,1.685820,16.204456,16.955235,15.269415
5,266.717607,-29.960362,2.608686,19.521484,20.976156,18.367470
6,266.720614,-29.950411,1.979046,20.799057,21.566879,19.587833
7,266.744969,-29.950876,2.702156,17.541964,19.052071,16.349915
8,266.740381,-29.942200,1.768490,17.089384,17.944401,16.175911
9,266.738246,-29.938497,2.594271,20.855574,22.206413,19.612143


## 11 · Exploratory data analysis

In [20]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nBasic statistics:')
display(df.describe())
print('\nNull counts:')
print(df.isnull().sum())

Shape: (1000, 4)

Column dtypes:
bp_rp               float32
phot_g_mean_mag     float32
phot_bp_mean_mag    float32
phot_rp_mean_mag    float32
dtype: object

Basic statistics:


,bp_rp,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,2.325176,18.987875,20.137308,17.812132
std,0.588090,1.662145,1.712748,1.586708
min,-0.830359,11.289200,12.613127,10.171391
25%,1.995394,18.096608,19.299670,16.918519
50%,2.263199,19.379299,20.710241,18.134757
75%,2.579528,20.284394,21.393608,19.028598
max,5.851952,21.244629,23.191965,21.030687



Null counts:
bp_rp               0
phot_g_mean_mag     0
phot_bp_mean_mag    0
phot_rp_mean_mag    0
dtype: int64


In [26]:
from display_html import generate_report

generate_report(df, USER_QUERY, output_path='gaia_report.html')

Building report ...
Report saved: gaia_report.html  (646 KB)
